# Male-specific contigs: which chromosome is the Y?

Contigs assembled from **male-specific 31-mers** were BLASTed against a **single-haplotype**
assembly with 13 chromosomes. This notebook filters the hits, classifies each contig by where
it lands, and counts the contigs per chromosome. No chromosome is labelled in advance: the one
that collects the most **single-hit** contigs, in total and per Mb, is the Y candidate.

## 1. Settings

In [ ]:
BLAST_TSV = "malehits.tsv"   # BLAST tabular output
# the fields of the -outfmt 6 string used for BLAST, in order
OUTFMT    = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore"
N_CHROM   = 13               # the N_CHROM longest scaffolds are the chromosomes

# a hit must pass all four filters
MIN_PIDENT = 95.0            # percent identity
MIN_QCOV   = 0.80            # aligned length / contig length
MAX_EVALUE = 1e-10
MIN_QLEN   = 50              # contig length (bp)

# genome-wide density plot (section 7)
WINDOW    = 10_000           # window size (bp)
ROLLING_K = 50               # rolling mean width (windows)

## 2. Load and filter the hits

Each row is one HSP. The chromosomes are the `N_CHROM` longest scaffolds in the file (printed
below, worth a check); any other scaffold is treated as unplaced.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
                     'font.size': 9, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'grid.linestyle': '--', 'grid.alpha': 0.3})
COLORS = {'single hit': '#1f77b4', 'repeated': '#2ca02c', 'multi': '#d62728'}
CLASSES = list(COLORS)

raw = pd.read_csv(BLAST_TSV, sep='\t', names=OUTFMT.split(), comment='#',
                  dtype={'qseqid': str, 'sseqid': str})
raw['qcov'] = raw['length'] / raw['qlen']
hits = raw[(raw['qlen'] >= MIN_QLEN) & (raw['pident'] >= MIN_PIDENT) &
           (raw['qcov'] >= MIN_QCOV) & (raw['evalue'] <= MAX_EVALUE)].copy()
hits['pos'] = (hits['sstart'] + hits['send']) / 2          # hit midpoint on the scaffold

chrom_len = raw.groupby('sseqid')['slen'].max().nlargest(N_CHROM)   # longest first
chroms = chrom_len.index

print(f"{len(raw):,} HSPs from {raw['qseqid'].nunique():,} contigs; "
      f"{len(hits):,} HSPs from {hits['qseqid'].nunique():,} contigs pass the filters")
print('Chromosomes:', ', '.join(chroms))

## 3. Classify each contig

| class | the contig's passing hits |
|---|---|
| **single hit** | one HSP on one scaffold: the cleanest placement |
| **repeated** | several HSPs, all on the same scaffold |
| **multi** | two or more scaffolds: not uniquely placeable |

Saved to `contig_classification.tsv`, together with the best-scoring scaffold of each contig
and every scaffold it hits.

In [ ]:
g = hits.groupby('qseqid')
best = hits.loc[g['bitscore'].idxmax()].set_index('qseqid')    # best-scoring HSP per contig

per_contig = pd.DataFrame({
    'n_scaffolds':   g['sseqid'].nunique(),
    'n_hsp':         g.size(),
    'best_scaffold': best['sseqid'],
    'best_bitscore': best['bitscore'],
    'all_scaffolds': g['sseqid'].agg(lambda s: ','.join(sorted(set(s)))),
})
per_contig['classification'] = np.select(
    [per_contig['n_scaffolds'] > 1, per_contig['n_hsp'] > 1], ['multi', 'repeated'], 'single hit')

per_contig.to_csv('contig_classification.tsv', sep='\t', index_label='contig')
per_contig['classification'].value_counts()

## 4. Per-chromosome summary

Contigs per chromosome by class (a `multi` contig counts on every chromosome it hits), and
single hits per Mb, because a longer chromosome collects more hits by chance alone. Sorted by
single hits, so the Y candidate should top the table. Saved to `chromosome_summary.tsv`.

In [ ]:
# one row per (contig, scaffold it hits)
touch = (per_contig.assign(scaffold=per_contig['all_scaffolds'].str.split(','))
                   .explode('scaffold', ignore_index=True))
summary = (pd.crosstab(touch['scaffold'], touch['classification'])
             .reindex(index=chroms, columns=CLASSES, fill_value=0)
             .rename_axis(index='chromosome', columns=None))
summary.insert(0, 'length_mb', chrom_len / 1e6)
summary['single_hit_per_mb'] = summary['single hit'] / summary['length_mb']
summary = summary.round(3).sort_values('single hit', ascending=False)
summary.to_csv('chromosome_summary.tsv', sep='\t')

unplaced = (per_contig['classification'] == 'single hit') & ~per_contig['best_scaffold'].isin(chroms)
print(f'{unplaced.sum():,} single-hit contigs fall on unplaced scaffolds and are not counted below')
summary

## 5. Where the single-hit contigs fall

One row per chromosome, longest on top, drawn to scale; one tick per single-hit contig, so
darker stretches hold more hits.

In [ ]:
single = per_contig.index[per_contig['classification'] == 'single hit']
sh = hits[hits['qseqid'].isin(single) & hits['sseqid'].isin(chroms)]    # one HSP per contig
row = {c: i for i, c in enumerate(chroms)}

fig, ax = plt.subplots(figsize=(12, 0.4 * len(chroms) + 1.5))
ax.hlines(range(len(chroms)), 0, chrom_len / 1e6, color='0.88', lw=4, zorder=1)
ax.scatter(sh['pos'] / 1e6, sh['sseqid'].map(row), marker='|', s=150, alpha=0.3,
           color=COLORS['single hit'], linewidths=1, zorder=2)
ax.set_yticks(range(len(chroms)), [f"{c}  (n={summary.at[c, 'single hit']:,})" for c in chroms])
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel('Position (Mb)')
ax.set_title(f'Single-hit male-specific contig positions ({len(sh):,} contigs)')
ax.grid(axis='x')
fig.savefig('single_hit_placements.png')
plt.show()

## 6. Counts per chromosome

Chromosomes ranked by single hits per Mb, highest first. Left: contigs per chromosome by class.
Right: single hits per Mb, labelled with the raw count and chromosome length.

In [ ]:
s = summary.sort_values('single_hit_per_mb', ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 0.35 * len(s) + 1.5), sharey=True)
s[CLASSES].plot.barh(stacked=True, ax=ax1, width=0.8, color=COLORS)
ax2.barh(s.index, s['single_hit_per_mb'], height=0.8, color=COLORS['single hit'])
for i, (d, n, mb) in enumerate(zip(s['single_hit_per_mb'], s['single hit'], s['length_mb'])):
    ax2.text(d, i, f"  n={n:,}, {mb:.0f} Mb", va='center', fontsize=7, color='0.3')
ax1.invert_yaxis()
ax1.set(xlabel='contigs', ylabel='chromosome', title='Contigs per chromosome')
ax2.set(xlabel='single-hit contigs per Mb', title='Single-hit contig density')
ax2.margins(x=0.22)
for ax in (ax1, ax2):
    ax.grid(axis='x')
ax1.legend(frameon=False)
fig.savefig('chromosome_counts.png')
plt.show()

## 7. Single hits along the genome

The chromosomes laid end to end, longest first: single-hit contigs per `WINDOW`-bp window
(alternating colours per chromosome) with their rolling mean over `ROLLING_K` windows. Windows
without hits count as 0. The y-axis is square-root scaled so a rolling mean far below 1 hit per
window is not flattened against the floor (0 stays 0).

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
offset = 0
for i, c in enumerate(chroms):
    counts = np.bincount((sh.loc[sh['sseqid'] == c, 'pos'] // WINDOW).astype(int),
                         minlength=int(chrom_len[c] // WINDOW) + 1)
    x = offset + np.arange(len(counts)) * WINDOW
    ax.scatter(x, counts, s=6, alpha=0.45, linewidths=0, rasterized=True,
               color=('#555555', '#4682B4')[i % 2])
    ax.plot(x, pd.Series(counts).rolling(ROLLING_K, center=True, min_periods=1).mean(),
            color='#E67E22', lw=0.8, alpha=0.9)
    offset += chrom_len[c]

ax.set_xticks(chrom_len.cumsum() - chrom_len / 2, chroms, rotation=45, ha='right')
ax.set_xlim(0, offset)
ax.set(xlabel='Chromosome', ylabel=f'M single hits / {WINDOW // 1000}kb',
       title='Male-specific k-mer contigs (single hit)')
ax.set_yscale('function', functions=(np.sqrt, np.square))    # sqrt axis: lifts small values
ax.set_ylim(bottom=0)
ax.grid(axis='y')
fig.savefig('single_hit_manhattan.png')
plt.show()